# 🔮 Step 4: Prediction & Analysis

Use the trained model to classify new images and calculate areas.

## What This Notebook Does:
- ✅ Load trained CNN model
- ✅ Load new/test satellite images
- ✅ Make predictions
- ✅ Visualize results
- ✅ Calculate area changes (km²)

---

**Previous:** [03_model_training.ipynb](03_model_training.ipynb)  
**Next:** [05_visualization_reports.ipynb](05_visualization_reports.ipynb)

## Load Model and Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import pickle
from tensorflow.keras.models import load_model
from matplotlib.colors import ListedColormap

# Load trained model
print("📂 Loading trained model...")
model = load_model('outputs/coastal_cnn_model.h5')

# Load metadata
with open('outputs/model_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

class_names = metadata['class_names']

print(f"✅ Model loaded!")
print(f"   Test Accuracy: {metadata['test_accuracy']*100:.2f}%")
print(f"   Classes: {list(class_names.values())}")

## Load New Images for Prediction

*For this demo, using the same images. In production, load different year's data.*

In [ ]:
# Load bands (for demo, using same images - replace with new year's data)
print("📂 Loading images...")
new_B02 = rasterio.open("coastalImage/B02.tiff").read(1).astype(float)
new_B03 = rasterio.open("coastalImage/B03.tiff").read(1).astype(float)
new_B04 = rasterio.open("coastalImage/B04.tiff").read(1).astype(float)
new_B08 = rasterio.open("coastalImage/B08.tiff").read(1).astype(float)

# Calculate NDVI
new_ndvi = (new_B08 - new_B04) / (new_B08 + new_B04 + 1e-10)

# Store shape for reshaping
original_shape = new_B02.shape

# Flatten and prepare
new_data = np.stack([
    new_B02.flatten(),
    new_B03.flatten(),
    new_B04.flatten(),
    new_B08.flatten(),
    new_ndvi.flatten()
], axis=1)

new_X = new_data.reshape(-1, 5, 1)

print(f"✅ Images loaded!")
print(f"   Shape: {original_shape}")
print(f"   Pixels: {new_X.shape[0]:,}")

## Make Predictions

In [ ]:
# Make predictions
print("🔮 Making predictions...")
predictions = model.predict(new_X, batch_size=1024, verbose=1)
pred_labels = np.argmax(predictions, axis=1)

# Reshape to image
pred_map = pred_labels.reshape(original_shape)

print(f"\n✅ Predictions complete!")

# Count classes
unique_pred, counts_pred = np.unique(pred_labels, return_counts=True)
print("\nPredicted Class Distribution:")
for label, count in zip(unique_pred, counts_pred):
    percentage = (count / pred_labels.size) * 100
    print(f"  {class_names[label]:15s}: {count:10,} pixels ({percentage:5.2f}%)")

## Visualize Predictions

In [ ]:
# Visualize predictions
colors = ['gray', 'green', 'blue', 'yellow']
cmap = ListedColormap(colors)

plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
im1 = plt.imshow(new_ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
plt.title('NDVI Map', fontsize=14, fontweight='bold')
plt.axis('off')
plt.colorbar(im1, label='NDVI')

plt.subplot(1, 2, 2)
im2 = plt.imshow(pred_map, cmap=cmap, vmin=0, vmax=3)
plt.title('CNN Predictions', fontsize=14, fontweight='bold')
plt.axis('off')
cbar = plt.colorbar(im2, ticks=[0, 1, 2, 3])
cbar.set_ticklabels(['Cloud', 'Seagrass', 'Water', 'Sand'])

plt.tight_layout()
plt.savefig('outputs/04_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Predictions saved to 'outputs/04_predictions.png'")

## Calculate Area by Class

In [ ]:
# Sentinel-2 pixel size (10m resolution)
pixel_size_m = 10
pixel_area_km2 = (pixel_size_m ** 2) / 1e6

# Calculate areas
area_km2 = {}
print("📏 Area Calculation:")
print("="*60)
for label, count in zip(unique_pred, counts_pred):
    area = count * pixel_area_km2
    area_km2[class_names[label]] = area
    print(f"{class_names[label]:15s}: {count:10,} pixels → {area:10.4f} km²")

total_area = sum(area_km2.values())
print("="*60)
print(f"{'Total':15s}: {sum(counts_pred):10,} pixels → {total_area:10.4f} km²")

# Bar chart
plt.figure(figsize=(10, 6))
classes = list(area_km2.keys())
areas = list(area_km2.values())
colors_plot = [colors[int(k)] for k in unique_pred]

bars = plt.bar(classes, areas, color=colors_plot, edgecolor='black', linewidth=2)
plt.ylabel('Area (km²)', fontsize=12)
plt.title('Coastal Area Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

for bar, area in zip(bars, areas):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{area:.2f}', ha='center', va='bottom', fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/04_area_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Save results
results = {
    'pred_map': pred_map,
    'area_km2': area_km2,
    'pixel_counts': dict(zip(unique_pred, counts_pred)),
    'total_area': total_area
}

with open('outputs/prediction_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("\n💾 Results saved!")
print("\n" + "="*60)
print("✅ PREDICTION & ANALYSIS COMPLETE!")
print("="*60)
print("\n📌 Next Step: Open 05_visualization_reports.ipynb for multi-year trends")